In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('D:\Internship\Vehicle_Price_Prediction_working\data\cleaned_data.csv')

In [3]:
df.head()

,name,description,make,model,year,price,engine,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain
0,2024 Jeep Wagoneer Series II,"\n \n Heated Leather Seats, Nav Sy...",Jeep,Wagoneer,2024,74600.0,24V GDI DOHC Twin Turbo,6,Gasoline,10.0,8-Speed Automatic,Series II,SUV,4,White,Global Black,Four-wheel Drive
1,2024 Jeep Grand Cherokee Laredo,Al West is committed to offering every custome...,Jeep,Grand Cherokee,2024,50170.0,OHV,6,Gasoline,1.0,8-Speed Automatic,Laredo,SUV,4,Metallic,Global Black,Four-wheel Drive
2,2024 GMC Yukon XL Denali,unknown,GMC,Yukon XL,2024,96410.0,"6.2L V-8 gasoline direct injection, variable v...",8,Gasoline,0.0,Automatic,Denali,SUV,4,Summit White,Teak/Light Shale,Four-wheel Drive
3,2023 Dodge Durango Pursuit,White Knuckle Clearcoat 2023 Dodge Durango Pur...,Dodge,Durango,2023,46835.0,16V MPFI OHV,8,Gasoline,32.0,8-Speed Automatic,Pursuit,SUV,4,White Knuckle Clearcoat,Black,All-wheel Drive
4,2024 RAM 3500 Laramie,\n \n 2024 Ram 3500 Laramie Billet...,RAM,3500,2024,81663.0,24V DDI OHV Turbo Diesel,6,Diesel,10.0,6-Speed Automatic,Laramie,Pickup Truck,4,Silver,Black,Four-wheel Drive


In [4]:
df.drop(columns=['name','description'],axis=1,inplace=True)

In [5]:
def mileage_bucket(x):
    if x>=0 and x<100:
        return 'brand new'
    elif x>100 and x<500:
        return 'test driven'
    elif x>500 and x<2000:
        return 'slightly used'
    elif x>2000 and x<5000:
        return 'early used'
    else:
        return 'light used demo'
    
df['mileage_bucket'] = df['mileage'].apply(mileage_bucket)

In [6]:
df['mileage'] = np.log1p(df['mileage'])

In [7]:
def simplified_transmission(x):
    x = str(x).lower()
    if 'cvt' in x:
        return 'cvt'
    elif 'dual' in x or 'dct' in x:
        return 'dual clutch'
    elif 'manual' in x or 'm/t' in x:
        return 'manual'
    elif '1-speed' in x or 'battery' in x or 'electric' in x:
        return 'single-speed (EV)'
    elif 'automatic' in x or 'a/t' in x:
        return 'automatic'
    else:
        return 'other'

df['transmission'] = df['transmission'].apply(simplified_transmission)

In [8]:
def extraction(x):
    x = str(x).lower()
    turbo = 1 if 'turbo' in 'supercharged' in x else 0

    if 'gdi' in x or 'pdi' in x:
        fuel_sys = 'direct injection'
    elif 'mpfi' in x:
        fuel_sys = 'multipoint injection'
    elif 'ddi' in x:
        fuel_sys = 'diesel direct injection'
    else:
        fuel_sys = 'other'

    return pd.Series([turbo, fuel_sys])

df[['turbo', 'fuel_sys',]] = df['engine'].apply(extraction)

In [9]:
df.head()

,make,model,year,price,engine,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain,mileage_bucket,turbo,fuel_sys
0,Jeep,Wagoneer,2024,74600.0,24V GDI DOHC Twin Turbo,6,Gasoline,2.397895,automatic,Series II,SUV,4,White,Global Black,Four-wheel Drive,brand new,0,direct injection
1,Jeep,Grand Cherokee,2024,50170.0,OHV,6,Gasoline,0.693147,automatic,Laredo,SUV,4,Metallic,Global Black,Four-wheel Drive,brand new,0,other
2,GMC,Yukon XL,2024,96410.0,"6.2L V-8 gasoline direct injection, variable v...",8,Gasoline,0.000000,automatic,Denali,SUV,4,Summit White,Teak/Light Shale,Four-wheel Drive,brand new,0,other
3,Dodge,Durango,2023,46835.0,16V MPFI OHV,8,Gasoline,3.496508,automatic,Pursuit,SUV,4,White Knuckle Clearcoat,Black,All-wheel Drive,brand new,0,multipoint injection
4,RAM,3500,2024,81663.0,24V DDI OHV Turbo Diesel,6,Diesel,2.397895,automatic,Laramie,Pickup Truck,4,Silver,Black,Four-wheel Drive,brand new,0,diesel direct injection


In [10]:
df['fuel'].value_counts()

fuel
Gasoline                634
Hybrid                  134
Electric                 94
Diesel                   72
PHEV Hybrid Fuel         15
E85 Flex Fuel             5
Diesel (B20 capable)      1
Name: count, dtype: int64

In [11]:
def fuel_merge(x):
    if x == 'PHEV Hybrid Fuel':
        return 'Hybrid'
    elif x == 'Diesel (B20 capable)':
        return 'Diesel'
    else:
        return x
    
df['fuel'] = df['fuel'].apply(fuel_merge)


In [12]:
df.drop(columns='engine',axis=1,inplace=True)

In [13]:
df.head()

,make,model,year,price,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain,mileage_bucket,turbo,fuel_sys
0,Jeep,Wagoneer,2024,74600.0,6,Gasoline,2.397895,automatic,Series II,SUV,4,White,Global Black,Four-wheel Drive,brand new,0,direct injection
1,Jeep,Grand Cherokee,2024,50170.0,6,Gasoline,0.693147,automatic,Laredo,SUV,4,Metallic,Global Black,Four-wheel Drive,brand new,0,other
2,GMC,Yukon XL,2024,96410.0,8,Gasoline,0.000000,automatic,Denali,SUV,4,Summit White,Teak/Light Shale,Four-wheel Drive,brand new,0,other
3,Dodge,Durango,2023,46835.0,8,Gasoline,3.496508,automatic,Pursuit,SUV,4,White Knuckle Clearcoat,Black,All-wheel Drive,brand new,0,multipoint injection
4,RAM,3500,2024,81663.0,6,Diesel,2.397895,automatic,Laramie,Pickup Truck,4,Silver,Black,Four-wheel Drive,brand new,0,diesel direct injection


In [14]:
df.to_csv(r'D:\Internship\Vehicle_Price_Prediction_working\data\featured_data.csv',index=False)